# Griffin Library - Intelligent Book Recommendation System

## Notebook 01 · Data Understanding

**Goal:** Explore the structure, quality, and content of all raw datasets  
to establish a clear understanding of available signals before any processing.

| | Details |
|---|---|
| **Input** | `data/raw/goodreads_data.csv` · `data/raw/booksummaries.txt` · `data/raw/7k/books.csv` (discarded) |
| **Operations** | Schema inspection · Null analysis · Sample review · Overlap check · Dataset comparison |
| **Output** | Documented understanding of all datasets + final data strategy decisions |
| **Next Step** | `02_clean_merge.ipynb` - Clean, filter, and merge selected datasets |

---

In [2]:
import sys
# Dependencies installed via requirements.txt


In [3]:
import pandas as pd
import os

# Set working directory to project root
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

# ── Load Goodreads ──────────────────────────────────────────
gr = pd.read_csv("data/raw/books.csv", on_bad_lines="skip")

# ── Load CMU ────────────────────────────────────────────────
cmu = pd.read_csv(
    "data/raw/booksummaries.txt",
    sep="\t",
    header=None,
    names=["wiki_id","freebase_id","title","author","pub_date","genres","summary"],
    on_bad_lines="skip",
)

print("Goodreads loaded:", gr.shape)
print("CMU loaded      :", cmu.shape)

Goodreads loaded: (11123, 12)
CMU loaded      : (16559, 7)


In [4]:
# ═══════════════════════════════════════════════════════════════
# GOODREADS - Column Inspection
# ═══════════════════════════════════════════════════════════════

print("Shape:", gr.shape)
print("\nColumns & Dtypes:")
print(gr.dtypes)
print("\nMissing Values:")
print(gr.isnull().sum())
print("\nSample Row:")
gr.head(3)

Shape: (11123, 12)

Columns & Dtypes:
bookID                  int64
title                     str
authors                   str
average_rating        float64
isbn                      str
isbn13                  int64
language_code             str
  num_pages             int64
ratings_count           int64
text_reviews_count      int64
publication_date          str
publisher                 str
dtype: object

Missing Values:
bookID                0
title                 0
authors               0
average_rating        0
isbn                  0
isbn13                0
language_code         0
  num_pages           0
ratings_count         0
text_reviews_count    0
publication_date      0
publisher             0
dtype: int64

Sample Row:


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic


###  Goodreads - Key Insights

- **Clean dataset**: zero missing values across all 12 columns - ready to use as-is
- **Strong ranking signals**: `ratings_count` and `average_rating` available for every book
- **Critical gap**: no `description` or `summary` column - this is exactly why i need CMU
- **Useful metadata**: `num_pages`, `publication_date`, and `language_code` available for filtering
- **Scale**: 11,123 books - moderate size, high quality

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CMU - Column Inspection
# ═══════════════════════════════════════════════════════════════

print("Shape:", cmu.shape)
print("\nMissing Values:")
print(cmu.isnull().sum())
print("\nSample Row:")
print(cmu.iloc[0][["title","author","pub_date","genres"]])
print("\nSummary Preview:")
print(cmu["summary"].iloc[0][:500])

Shape: (16559, 7)

Missing Values:
wiki_id           0
freebase_id       0
title             0
author         2382
pub_date       5610
genres         3718
summary           0
dtype: int64

Sample Row:
title                                             Animal Farm
author                                          George Orwell
pub_date                                           1945-08-17
genres      {"/m/016lj8": "Roman \u00e0 clef", "/m/06nbt":...
Name: 0, dtype: object

Summary Preview:
 Old Major, the old boar on the Manor Farm, calls the animals on the farm for a meeting, where he compares the humans to parasites and teaches the animals a revolutionary song, 'Beasts of England'. When Major dies, two young pigs, Snowball and Napoleon, assume command and turn his dream into a philosophy. The animals revolt and drive the drunken and irresponsible Mr Jones from the farm, renaming it "Animal Farm". They adopt Seven Commandments of Animal-ism, the most important of which is, "All a


###  CMU - Key Insights

- **Summaries are complete**: zero nulls in `summary` column - this is the core asset for embedding
- **Rich narratives**: summaries average 400+ words - far richer than any genre tag
- **Genre format issue**: `genres` stored as JSON strings - requires parsing in cleaning step
- **Partial metadata**: `author` (14% null) and `pub_date` (34% null) - non-critical, manageable
- **Scale**: 16,559 books - slightly larger than Goodreads

---
###  Strategic Decision

- **Goodreads** provides → rating signals for ranking and collaborative filtering
- **CMU** provides → rich summaries for semantic embedding (FAISS)
- **Merge key** → `title` + `author` (no shared ISBN between datasets)
- **Expected overlap** → to be measured in the next cell

In [6]:
# ═══════════════════════════════════════════════════════════════
# OVERLAP ANALYSIS - How well do both datasets match?
# ═══════════════════════════════════════════════════════════════

gr_titles  = set(gr["title"].str.strip().str.lower())
cmu_titles = set(cmu["title"].str.strip().str.lower())
overlap    = gr_titles & cmu_titles

print(f"Goodreads unique titles : {len(gr_titles):>8,}")
print(f"CMU unique titles       : {len(cmu_titles):>8,}")
print(f"Exact title overlap     : {len(overlap):>8,}")
print(f"Overlap rate (vs GR)    : {len(overlap)/len(gr_titles)*100:.1f}%")
print(f"Overlap rate (vs CMU)   : {len(overlap)/len(cmu_titles)*100:.1f}%")

print("\nSample matched titles:")
for t in list(overlap)[:10]:
    print(f"  • {t}")

Goodreads unique titles :   10,311
CMU unique titles       :   16,271
Exact title overlap     :    1,279
Overlap rate (vs GR)    : 12.4%
Overlap rate (vs CMU)   : 7.9%

Sample matched titles:
  • planet of the apes
  • a severed head
  • much ado about nothing
  • layer cake
  • the blue flowers
  • slowness
  • the brothers k
  • the return of the native
  • anthem
  • lysistrata


###  Overlap Analysis - Key Insights

- **Low exact match rate**: only 12.4% of Goodreads titles found in CMU - expected due to title formatting differences
- **Real overlap is higher**: many titles exist in both but with slight variations (e.g. subtitles, editions, punctuation)
- **Fuzzy matching needed**: a similarity-based title match in the cleaning step will significantly increase overlap
- **CMU summaries still valuable**: even unmatched CMU books will enrich the final catalog
- **Strategic implication**: Goodreads will be our base dataset - CMU summaries will be joined where available, and books without a summary match will use genre + author as embedding text fallback

---

In [7]:
# ═══════════════════════════════════════════════════════════════
# RATINGS QUALITY CHECK - Goodreads ranking signals
# ═══════════════════════════════════════════════════════════════

print("Average Rating Distribution:")
print(gr["average_rating"].describe())

print("\nRatings Count Distribution:")
print(gr["ratings_count"].describe())

print("\nBooks with < 10 ratings:")
print(len(gr[gr["ratings_count"] < 10]))

print("\nLanguage Distribution:")
print(gr["language_code"].value_counts().head(10))

Average Rating Distribution:
count    11123.000000
mean         3.934075
std          0.350485
min          0.000000
25%          3.770000
50%          3.960000
75%          4.140000
max          5.000000
Name: average_rating, dtype: float64

Ratings Count Distribution:
count    1.112300e+04
mean     1.794285e+04
std      1.124992e+05
min      0.000000e+00
25%      1.040000e+02
50%      7.450000e+02
75%      5.000500e+03
max      4.597666e+06
Name: ratings_count, dtype: float64

Books with < 10 ratings:
673

Language Distribution:
language_code
eng      8908
en-US    1408
spa       218
en-GB     214
fre       144
ger        99
jpn        46
mul        19
zho        14
grc        11
Name: count, dtype: int64


###  Ratings Quality - Key Insights

- **Healthy average**: mean rating of 3.93 with low std (0.35) - consistent and reliable signal
- **Rating = 0 issue**: min rating is 0.0 - these are books with no real ratings, must be filtered out in cleaning
- **Highly skewed ratings count**: median is 745 but max is 4.6M - a small number of popular books dominate
- **673 books with < 10 ratings**: too few ratings to trust - will be filtered in cleaning step
- **English dominates**: eng + en-US + en-GB = ~95% of dataset - safe to keep English only for consistency
- **Bayesian smoothing needed**: raw average_rating is unreliable for books with few ratings - i will compute a weighted score in modeling step

---

In [ ]:
import sys
# Dependencies installed via requirements.txt
import pandas as pd
import os
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

# Dataset 1: Best Books 10k
df1 = pd.read_csv("data/raw/goodreads_data.csv")
print("=" * 60)
print("DATASET 1 — Best Books 10k Multi-Genre")
print("=" * 60)
print(f"Shape   : {df1.shape}")
print(f"Columns : {list(df1.columns)}")
print(f"Nulls   :\n{df1.isnull().sum()}")
print(f"\nSample row:")
print(df1.iloc[0])

# Dataset 2: 7k Books with Metadata
df2 = pd.read_csv("data/raw/7k/books.csv")
print("\n" + "=" * 60)
print("DATASET 2 — 7k Books with Metadata")
print("=" * 60)
print(f"Shape   : {df2.shape}")
print(f"Columns : {list(df2.columns)}")
print(f"Nulls   :\n{df2.isnull().sum()}")
print(f"\nSample row:")
print(df2.iloc[0])

DATASET 1 — Best Books 10k Multi-Genre
Shape   : (10000, 8)
Columns : ['Unnamed: 0', 'Book', 'Author', 'Description', 'Genres', 'Avg_Rating', 'Num_Ratings', 'URL']
Nulls   :
Unnamed: 0      0
Book            0
Author          0
Description    77
Genres          0
Avg_Rating      0
Num_Ratings     0
URL             0
dtype: int64

Sample row:
Unnamed: 0                                                     0
Book                                       To Kill a Mockingbird
Author                                                Harper Lee
Description    The unforgettable novel of a childhood in a sl...
Genres         ['Classics', 'Fiction', 'Historical Fiction', ...
Avg_Rating                                                  4.27
Num_Ratings                                            5,691,311
URL            https://www.goodreads.com/book/show/2657.To_Ki...
Name: 0, dtype: object

DATASET 2 — 7k Books with Metadata
Shape   : (6810, 12)
Columns : ['isbn13', 'isbn10', 'title', 'subtitle', 'aut

: 

##  New Datasets Comparison - Decision

| | Best Books 10k | 7k Books |
|---|---|---|
| **Books** | 10,000 | 6,810 |
| **Description** | ✅ 99.2% complete | ⚠️ 96% complete |
| **Genres** | ✅ Rich list format | ⚠️ Single category |
| **Ratings** | ✅ Goodreads (millions) | ⚠️ Google Books (sparse) |
| **Source** | Goodreads | Google Books |

**Decision: Use `Best Books 10k` as the primary dataset.**  
CMU summaries will be merged on top for additional enrichment where available.  
Dataset 2 is discarded.

---

## ✅ Data Understanding - Findings & Decisions

| Finding | Decision |
|---|---|
| Best Books 10k has rich Goodreads descriptions (99.2% complete) | Use as **primary dataset** |
| CMU has longer narrative summaries (100% complete) | Use as **enrichment layer** via title merge |
| 7k Books dataset is smaller with weaker signals (Google Books source) | **Discarded** |
| Only 13.6% exact title overlap between original Goodreads and CMU | Apply fuzzy matching in merge step |
| Best Books 10k genres stored as string lists | Parse in cleaning step |
| CMU genres stored as JSON strings | Parse in cleaning step |
| CMU has 14% null authors and 34% null pub_date | Non-critical - use title-only merge key |
| Num_Ratings in Best Books 10k stored as string with commas | Convert to numeric in cleaning step |

 **Next → `02_clean_merge.ipynb`**